In [32]:
import asyncio
import os
import time
import base64
import mimetypes
from pathlib import Path

import json

from dotenv import load_dotenv
from google import genai
from IPython.display import Image, display

import httpx
from pprint import pprint

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")


model = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(api_key=api_key)
print('준비 완료 / 사용 모델: ',model)
async_client = client.aio

access_token = os.getenv('tmdb_token')

header = { 
    "Authorization": f"Bearer {access_token}"
}

system_instruction = """
TMDB API의 원본 JSON을 그대로 출력하지 않는다.
사용자의 질문에 필요한 정보만 자연어로 정리한다.
영화 제목, 개봉일, 평점, 줄거리 등 질문과 관련된 필드를 사용한다.
검색 결과가 없으면 찾지 못했다는 사실을 알린다.
API 호출에 실패하면 임의의 영화 정보를 만들지 않고 오류를 안내한다.
사용자가 종료한다고 하면 서비스 종료한다고 하고 끝낸다.
"""



준비 완료 / 사용 모델:  gemini-3.5-flash-lite


In [ ]:

async def get_post(client: httpx.AsyncClient, option: str, page: int = 1):

    url = f"https://api.themoviedb.org/3/movie/{option}"

    response = await client.get(url,headers=header,params={'page':page})
    
    response.raise_for_status

    output = response.json()

    return output

async def get_searching_movie(client: httpx.AsyncClient, search: str, page: int = 1):

    url = "https://api.themoviedb.org/3/search/movie"

    response = await client.get(url,headers=header,params={'query':search,'page':page})
    
    response.raise_for_status

    output = response.json()

    return output

async def get_credits(client: httpx.AsyncClient, movie_id: int):

    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"

    response = await client.get(url,headers=header)

    response.raise_for_status

    output = response.json()

    return output



In [ ]:

async def get_movie_list(option: str,pages: int):

    options = {'now_playing','popular','top_rated','upcoming'}

    if option not in options:
            return {"ok": False, "error": f"허용되지 않은 option입니다: {option}"}

    try:

        async with httpx.AsyncClient() as http_client:
             page_requests = [get_post(http_client,option,page) for page in range(1,pages+1)]
             responses = await asyncio.gather(*page_requests)

        movies = [movie for result in responses for movie in result['results']]

        return {"ok": True, "data": movies}

    except Exception as e:
        return {"ok": False, "error": f"영화 목록 조회 실패: {e}"}



get_movie_list_tool = {
    "type": "function",
    "name": "get_movie_list_tool",
    "description": "인기작, 현재 상영작, 평점순, 개봉예정작 등 영화 목록을 TMDB API에서 조회한다. 여러 페이지를 동시에 요청해 결과를 합쳐서 반환한다.",
    "parameters": {
        "type": "object",
        "properties": {
            "option": {
                "type": "string",
                "enum": ["now_playing", "popular", "top_rated", "upcoming"],
                "description": "조회할 영화 목록의 종류 (현재 상영중/인기/평점순/개봉예정)"
            },
            "pages": {
                "type": "integer",
                "minimum": 2,
                "description": "조회할 페이지 수 (최소 2페이지 이상)"
            }
        },
        "required": ["option", "pages"]
    }
}


In [ ]:
async def get_movie_detail(movie_id: int):
    try:
        movie_id = int(movie_id)

        async with httpx.AsyncClient() as http_client:
                detail, credits = await asyncio.gather(
                    get_post(http_client, movie_id),
                    get_credits(http_client, movie_id),
                )

        cast = [
            {"name": member["name"], "character": member.get("character")}
            for member in credits.get("cast", [])[:10]
        ]

        return {"ok": True, "data": {**detail, "cast": cast}}

    except (ValueError, TypeError):
        return {"ok": False, "error": "movie_id는 정수여야 합니다."}

    except Exception as e:
        return {"ok": False, "error": f"영화 조회 실패: {e}"}
    

get_movie_detail_tool = {
    "type": "function",
    "name": "get_movie_detail_tool",
    "description": "영화 ID로 특정 영화의 상세 정보(제목, 개요, 평점, 개봉일, 주요 출연진 등)를 조회합니다. 영화 목록 조회나 검색 결과에서 얻은 id 값을 movie_id로 사용해야 합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "movie_id": {
                "type": "integer",
                "description": "상세 정보를 조회할 영화의 고유 ID."
            }
        },
        "required": ["movie_id"]
    }
}



In [36]:
async def search_movies(search,pages):

    if not search:
            return {"ok": False, "error": "검색할 영화 제목이 없습니다."}
    try:

        async with httpx.AsyncClient() as http_client:
             page_requests = [get_searching_movie(http_client,search,page) for page in range(1,pages+1)]
             responses = await asyncio.gather(*page_requests)

        movies = [movie for result in responses for movie in result['results']]

        return {"ok": True, "data": movies}

    except Exception as e:
        return {"ok": False, "error": f"영화 검색 실패: {e}"}

search_movies_tool = {
    "type": "function",
    "name": "search_movies_tool",
    "description": "영화 제목으로 영화를 검색합니다. 여러 페이지의 검색 결과를 한 번에 가져올 수 있습니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "search": {"type": "string", "description": "검색할 영화 제목 또는 키워드"},
            "pages": {"type": "integer", "description": "가져올 검색 결과 페이지 수"}
        },
        "required": ["search", "pages"]
    }
}

In [ ]:
async def get_top_rated_movies(option: str, pages: int, result_count: int, min_vote_count: int):

    options = {'now_playing', 'popular', 'top_rated', 'upcoming'}

    if option not in options:
        return {"ok": False, "error": f"허용되지 않은 option입니다: {option}"}

    try:
        async with httpx.AsyncClient() as http_client:
            page_requests = [get_post(http_client, option, page) for page in range(1, pages + 1)]
            responses = await asyncio.gather(*page_requests)

        movies = [movie for result in responses for movie in result['results']]
        filtered = [movie for movie in movies if movie.get('vote_count', 0) >= min_vote_count]
        top_movies = sorted(filtered, key=lambda movie: movie['vote_average'], reverse=True)[:result_count]

        return {"ok": True, "data": top_movies}

    except Exception as e:
        return {"ok": False, "error": f"평점 상위 영화 조회 실패: {e}"}


get_top_rated_movies_tool = {
    "type": "function",
    "name": "get_top_rated_movies_tool",
    "description": "영화 목록(현재 상영작/인기작/평점순/개봉예정작)을 여러 페이지 조회한 뒤, 평점과 평가 수를 기준으로 상위 영화를 선별해 반환한다.",
    "parameters": {
        "type": "object",
        "properties": {
            "option": {
                "type": "string",
                "enum": ["now_playing", "popular", "top_rated", "upcoming"],
                "description": "조회할 영화 목록의 종류 (현재 상영중/인기/평점순/개봉예정)"
            },
            "pages": {
                "type": "integer",
                "description": "조회할 페이지 수"
            },
            "result_count": {
                "type": "integer",
                "description": "반환할 상위 영화 개수"
            },
            "min_vote_count": {
                "type": "integer",
                "description": "선별 기준이 되는 최소 평가 수 (이보다 적은 평가를 받은 영화는 제외)"
            }
        },
        "required": ["option", "pages", "result_count", "min_vote_count"]
    }
}

In [ ]:
TOOLS = [get_movie_list_tool,get_movie_detail_tool,search_movies_tool,get_top_rated_movies_tool]
TOOL_FUNCTIONS = {
                'get_movie_list_tool':get_movie_list,
                'get_movie_detail_tool':get_movie_detail,
                'search_movies_tool':search_movies,
                'get_top_rated_movies_tool':get_top_rated_movies,
            }


In [ ]:


async def execute_tool_call(step) -> dict:
    tool_function = TOOL_FUNCTIONS.get(step.name)
    if tool_function is None:
        return {"ok": False, "error": f"허용되지 않은 도구: {step.name}"}

    try:
        return await tool_function(**step.arguments)
    except TypeError as error:
        return {"ok": False, "error": f"잘못된 인자: {error}"}
    except Exception as error:
        return {"ok": False, "error": f"도구 실행 실패: {type(error).__name__}"}
    

async def run_agent(user_input: str, previous_interaction_id: str = None, max_turns: int = 5) -> dict:
    next_input = user_input
    logs = []

    for turn in range(1, max_turns + 1):
        request = {
            "model": model,
            "input": next_input,
            "system_instruction": system_instruction,
            "tools": TOOLS,
            "store": True,
        }
        if previous_interaction_id is not None:
            request["previous_interaction_id"] = previous_interaction_id

        interaction = client.interactions.create(**request)
        function_calls = [step for step in interaction.steps if step.type == "function_call"]

        if not function_calls:
            return {
                "ok": True,
                "answer": interaction.output_text,
                "turns": turn,
                "tool_logs": logs,
                "previous_interaction_id": interaction.id,
            }

        concurrent_answer = await asyncio.gather(*(execute_tool_call(step) for step in function_calls))

        next_input = []
        for step, result in zip(function_calls, concurrent_answer):
            logs.append({"turn": turn, "tool": step.name, "arguments": step.arguments, "result": result})
            next_input.append({
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}],
            })

        previous_interaction_id = interaction.id

    return {
        "ok": False,
        "answer": None,
        "turns": max_turns,
        "tool_logs": logs,
        "error": "최대 반복 횟수를 초과했습니다.",
        "previous_interaction_id": previous_interaction_id,
    }

    

In [ ]:
EXIT_WORDS = {"종료", "exit", "quit", "그만"}

previous_interaction_id = None

while True:
    user_input = input("나: ")

    if user_input.strip() in EXIT_WORDS:
        print("서비스를 종료합니다.")
        break

    result = await run_agent(user_input, previous_interaction_id=previous_interaction_id)

    print(result["answer"] if result["ok"] else result["error"])
    print("\n도구 실행 기록")
    pprint(result["tool_logs"])

    if result["ok"]:
        previous_interaction_id = result["previous_interaction_id"]
